# Send로 동적 팬아웃

검색된 문서 수가 실행할 때마다 달라도 각 문서를 같은 노드로 보내고 결과를 모을 수 있을까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 실행합니다. 설치 셀은 이 노트북에 필요한 버전을 고정합니다.

## 패키지 설치와 버전 확인

필요한 패키지를 고정 버전으로 설치합니다. 출력에서 LangGraph 버전 또는 API 키의 존재 여부만 확인합니다.

In [1]:
# 노트북을 독립 실행할 수 있도록 패키지와 환경을 준비합니다.
%pip install -q langgraph==1.2.11 langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 scikit-learn==1.9.0
from importlib.metadata import version
print('Python 패키지 준비 완료')
print('langgraph', version('langgraph'))

Note: you may need to restart the kernel to use updated packages.
Python 패키지 준비 완료
langgraph 1.2.11


## 전체 상태와 작업 상태 정의

다음 코드에서 전체 상태와 작업 상태 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

In [2]:
# 전체 상태와 작업 상태 정의에 사용할 상태 형식과 함수를 선언합니다.
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class OverallState(TypedDict):
    documents: list[dict]
    results: Annotated[list[dict], operator.add]

class DocumentState(TypedDict):
    document: dict

## Send 목록과 작업 노드 정의

다음 코드에서 Send 목록과 작업 노드 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

In [3]:
# Send 목록과 작업 노드 정의에 사용할 상태 형식과 함수를 선언합니다.
def dispatch(state: OverallState) -> list[Send]:
    return [Send('inspect_document', {'document': doc}) for doc in state['documents']]

def inspect_document(state: DocumentState) -> dict:
    doc = state['document']
    return {'results': [{'id': doc['id'], 'length': len(doc['text'])}]}

builder = StateGraph(OverallState)
builder.add_node('inspect_document', inspect_document)
builder.add_conditional_edges(START, dispatch)
builder.add_edge('inspect_document', END)
graph = builder.compile()

## 세 문서로 팬아웃

세 문서로 팬아웃 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

In [4]:
# 세 문서로 팬아웃 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
documents = [
    {'id': 'E01', 'text': '대출은 7권까지 가능합니다.'},
    {'id': 'E02', 'text': '대출 기간은 14일입니다.'},
    {'id': 'E03', 'text': '반납함은 출입구에 있습니다.'},
]
result = graph.invoke({'documents': documents, 'results': []})
ordered = sorted(result['results'], key=lambda item: item['id'])
print(ordered)
assert len(ordered) == 3

[{'id': 'E01', 'length': 15}, {'id': 'E02', 'length': 14}, {'id': 'E03', 'length': 15}]


## 입력 수 하나로 비교

입력 수 하나로 비교 조건으로 값을 하나만 변경합니다. 나란히 출력된 두 결과에서 달라진 항목을 확인합니다.

In [5]:
# 한 조건만 바꾸고 나머지 입력을 유지해 결과 차이를 비교합니다.
single = graph.invoke({'documents': documents[:1], 'results': []})
print(single['results'])
assert len(single['results']) == 1

[{'id': 'E01', 'length': 15}]
